# Zerodha Kite Circuit Breaker Trading Bot

## Overview
This notebook demonstrates how to create an automated trading bot that:
- Monitors stocks for circuit breaker levels
- Automatically places BUY orders when lower circuit is hit
- Automatically places SELL orders when upper circuit is hit
- Manages positions with Stop Loss and Take Profit
- Only trades during the first 3 hours (9:15 AM - 12:15 PM IST)

### What are Circuit Breakers?
- **Upper Circuit**: Stock rises 20% - signal to SELL (overvalued)
- **Lower Circuit**: Stock falls 20% - signal to BUY (undervalued)

## Step 1: Installation & Setup

In [ ]:
# Required libraries - ALL ALREADY INSTALLED! ✓
pip install kiteconnect schedule python-dotenv

# Import libraries that have been installed
from datetime import datetime, time
from typing import Dict, List, Tuple
import json

print("✅ Python Libraries Loaded Successfully")
print("-" * 60)

# Verify all required libraries are available
required_libs = {
    'kiteconnect': 'Zerodha Kite API',
    'schedule': 'Task scheduling',
    'dotenv': 'Environment variables'
}

for lib, description in required_libs.items():
    try:
        __import__(lib)
        print(f"  ✅ {lib:<20} - {description}")
    except ImportError:
        print(f"  ❌ {lib:<20} - NOT INSTALLED")

print("-" * 60)
print("✓ Ready to connect to Zerodha Kite!")
print("✓ Next: Configure API credentials in Step 2")

## Step 2: Kite API Authentication

### Getting Your API Credentials:

1. **Create App at**: https://developers.kite.trade/
2. **Get API Key**: From your app details
3. **Get Access Token**: 
   - Generate via login redirect
   - Keep these secret and safe!

In [ ]:
import os
from dotenv import load_dotenv

# Load credentials from .env file
load_dotenv()
API_KEY = os.getenv('KITE_API_KEY', 'not_set')
ACCESS_TOKEN = os.getenv('KITE_ACCESS_TOKEN', 'not_set')

print("="*60)
print("KITE API CONFIGURATION")
print("="*60)
print(f"✓ API Key loaded from .env:      {API_KEY != 'not_set'}")
print(f"✓ Access Token loaded from .env: {ACCESS_TOKEN != 'not_set'}")

if API_KEY == 'not_set' or ACCESS_TOKEN == 'not_set':
    print("\n⚠️  IMPORTANT SETUP STEPS:")
    print("1. Create .env file in this folder with:")
    print("   KITE_API_KEY=your_api_key")
    print("   KITE_ACCESS_TOKEN=your_access_token")
    print("\n2. Or set environment variables:")
    print("   $env:KITE_API_KEY='your_api_key'")
    print("   $env:KITE_ACCESS_TOKEN='your_access_token'")
    print("\n3. Then re-run this cell")
else:
    print("\n✅ Credentials are configured!")
    
    # Try to connect to Kite API
    try:
        from kiteconnect import KiteConnect
        kite = KiteConnect(api_key=API_KEY)
        kite.set_access_token(ACCESS_TOKEN)
        print("✅ Successfully connected to Zerodha Kite API!")
    except Exception as e:
        print(f"❌ Error connecting to Kite API: {e}")
        print("   Check your API credentials and try again.")

## Step 3: Understanding Circuit Breakers

Let's understand how circuit breakers work with an example:

In [ ]:
# Example: TCS Stock Circuit Breaker Calculation

previous_close = 3000  # Previous day closing price
upper_circuit_percent = 20
lower_circuit_percent = 20

# Calculate circuit levels
upper_circuit = previous_close * (1 + upper_circuit_percent / 100)
lower_circuit = previous_close * (1 - lower_circuit_percent / 100)

print("="*50)
print("CIRCUIT BREAKER EXAMPLE - TCS STOCK")
print("="*50)
print(f"Previous Close Price: Rs. {previous_close}")
print(f"\nUpper Circuit Level (20% up): Rs. {upper_circuit}")
print(f"Lower Circuit Level (20% down): Rs. {lower_circuit}")
print(f"\nTrading Range: Rs. {lower_circuit} - Rs. {upper_circuit}")

# Simulate different prices
prices = [2400, 2500, 2600, 3000, 3400, 3500, 3600]

print(f"\n{'Current Price':<15} {'% Change':<15} {'Action':<15}")
print("-"*50)

for price in prices:
    percent_change = ((price - previous_close) / previous_close) * 100
    
    if price >= upper_circuit * 0.98:
        action = "🔴 SELL (Upper)"
    elif price <= lower_circuit * 1.02:
        action = "🟢 BUY (Lower)"
    else:
        action = "⚪ Monitor"
    
    print(f"Rs. {price:<13} {percent_change:>+6.1f}% {action:<15}")

## Step 4: Trading Logic & Strategy

### Trading Rules:

| Event | Action | Stop Loss | Take Profit |
|-------|--------|-----------|-------------|
| **Lower Circuit Hit** (Stock ⬇️ 20%) | **BUY** (Go Long) | -2% | +5% |
| **Upper Circuit Hit** (Stock ⬆️ 20%) | **SELL** (Go Short) | +2% | -5% |
| **Market Close** (3:30 PM) | **Close all positions** | - | - |

### Example Trade Scenario:

In [ ]:
# Scenario 1: Lower Circuit Hit (BUY Signal)
print("="*60)
print("SCENARIO 1: LOWER CIRCUIT HIT - BUY TRADE")
print("="*60)

stock = "INFY"
previous_close = 1500
lower_circuit_price = previous_close * 0.80  # 20% down = 1200
current_price = 1200

quantity = 1
stop_loss = current_price * 0.98  # -2% = 1176
take_profit = current_price * 1.05  # +5% = 1260

print(f"Stock: {stock}")
print(f"Previous Close: Rs. {previous_close}")
print(f"Lower Circuit Level: Rs. {lower_circuit_price:.2f}")
print(f"\n→ Current Price: Rs. {current_price} (HIT LOWER CIRCUIT!)")
print(f"\n📊 BUY ORDER PLACED:")
print(f"   Quantity: {quantity} share")
print(f"   Entry Price: Rs. {current_price}")
print(f"   Stop Loss: Rs. {stop_loss:.2f} (Exit if falls 2% more)")
print(f"   Take Profit: Rs. {take_profit:.2f} (Exit if rises 5%)")

# Example exit scenarios
print(f"\n📈 Exit Scenarios:")
exit_prices = [1176, 1200, 1260]
for exit_price in exit_prices:
    if exit_price <= stop_loss:
        profit_loss = (exit_price - current_price) * quantity
        print(f"   At Rs. {exit_price}: ❌ STOP LOSS hit | P&L: Rs. {profit_loss:.2f}")
    elif exit_price >= take_profit:
        profit_loss = (exit_price - current_price) * quantity
        print(f"   At Rs. {exit_price}: ✅ TAKE PROFIT hit | P&L: Rs. {profit_loss:.2f}")

print("\n" + "="*60)
print("SCENARIO 2: UPPER CIRCUIT HIT - SELL TRADE")
print("="*60)

previous_close = 2000
upper_circuit_price = previous_close * 1.20  # 20% up = 2400
current_price = 2400

stop_loss = current_price * 1.02  # +2% = 2448
take_profit = current_price * 0.95  # -5% = 2280

print(f"Stock: RELIANCE")
print(f"Previous Close: Rs. {previous_close}")
print(f"Upper Circuit Level: Rs. {upper_circuit_price:.2f}")
print(f"\n→ Current Price: Rs. {current_price} (HIT UPPER CIRCUIT!)")
print(f"\n📊 SELL ORDER PLACED (SHORT):")
print(f"   Quantity: 1 share")
print(f"   Entry Price: Rs. {current_price}")
print(f"   Stop Loss: Rs. {stop_loss:.2f} (Exit if rises 2% more)")
print(f"   Take Profit: Rs. {take_profit:.2f} (Exit if falls 5%)")

print(f"\n📉 Exit Scenarios:")
exit_prices = [2448, 2400, 2280]
for exit_price in exit_prices:
    if exit_price >= stop_loss:
        profit_loss = (current_price - exit_price) * quantity
        print(f"   At Rs. {exit_price}: ❌ STOP LOSS hit | P&L: Rs. {profit_loss:.2f}")
    elif exit_price <= take_profit:
        profit_loss = (current_price - exit_price) * quantity
        print(f"   At Rs. {exit_price}: ✅ TAKE PROFIT hit | P&L: Rs. {profit_loss:.2f}")


## Step 5: Complete Circuit Trader Implementation

Here's the main CircuitTrader class that handles all trading logic:

In [ ]:
from datetime import datetime, time
from typing import Dict, List, Tuple

class CircuitTraderSimulator:
    """
    Simulator version of CircuitTrader for testing and understanding
    (Production version uses actual Kite API)
    """
    
    def __init__(self):
        self.trading_start = time(9, 15)
        self.trading_end = time(12, 15)
        self.upper_circuit = 20
        self.lower_circuit = 20
        self.quantity_per_trade = 1
        self.stop_loss_percent = 2
        self.take_profit_percent = 5
        self.positions = {}
        self.order_log = []
    
    def is_within_trading_window(self) -> bool:
        """Check if current time is within first 3 hours of trading"""
        current_time = datetime.now().time()
        is_weekday = datetime.now().weekday() < 5
        return is_weekday and self.trading_start <= current_time <= self.trading_end
    
    def calculate_circuit_levels(self, symbol: str, ltp: float, 
                                previous_close: float) -> Dict:
        """Calculate circuit breaker levels"""
        upper_level = previous_close * (1 + self.upper_circuit / 100)
        lower_level = previous_close * (1 - self.lower_circuit / 100)
        change_percent = ((ltp - previous_close) / previous_close) * 100
        
        return {
            'symbol': symbol,
            'ltp': ltp,
            'previous_close': previous_close,
            'upper_circuit': upper_level,
            'lower_circuit': lower_level,
            'change_percent': change_percent
        }
    
    def check_circuit_status(self, symbol: str, ltp: float, 
                            previous_close: float) -> Tuple[str, float]:
        """
        Check if stock hit circuit breaker
        Returns: ('UPPER', price), ('LOWER', price), or ('NONE', price)
        """
        levels = self.calculate_circuit_levels(symbol, ltp, previous_close)
        
        if ltp >= levels['upper_circuit'] * 0.98:
            return 'UPPER', ltp
        elif ltp <= levels['lower_circuit'] * 1.02:
            return 'LOWER', ltp
        else:
            return 'NONE', ltp
    
    def place_order(self, symbol: str, order_type: str, price: float):
        """Simulate placing an order"""
        order = {
            'symbol': symbol,
            'type': order_type,
            'price': price,
            'quantity': self.quantity_per_trade,
            'timestamp': datetime.now()
        }
        self.order_log.append(order)
        print(f"✓ {order_type} Order placed for {symbol} at Rs. {price}")
        return order
    
    def get_order_history(self):
        """Get all orders placed"""
        return self.order_log
    
    def simulate_trade(self, symbol: str, previous_close: float, 
                      prices: List[float]):
        """Simulate a trading scenario with price history"""
        print(f"\n{'='*70}")
        print(f"SIMULATION: {symbol}")
        print(f"{'='*70}")
        print(f"Previous Close: Rs. {previous_close}")
        
        for ltp in prices:
            circuit, _ = self.check_circuit_status(symbol, ltp, previous_close)
            levels = self.calculate_circuit_levels(symbol, ltp, previous_close)
            
            status = ""
            if circuit == 'UPPER':
                status = "🔴 UPPER CIRCUIT - SELL"
            elif circuit == 'LOWER':
                status = "🟢 LOWER CIRCUIT - BUY"
            else:
                status = "⚪ Normal trading"
            
            print(f"Price: Rs. {ltp:>7} | Change: {levels['change_percent']:>+6.1f}% | {status}")

# Test the simulator
print("Testing Circuit Breaker Detection\\n")

simulator = CircuitTraderSimulator()

# Simulation 1: INFY - Lower Circuit
print("SIMULATION 1: INFY Stock - Lower Circuit Scenario")
simulator.simulate_trade(
    'INFY', 
    previous_close=1500,
    prices=[1500, 1400, 1300, 1200, 1150]  # 1200 is lower circuit
)

# Simulation 2: TCS - Upper Circuit
print("\n\nSIMULATION 2: TCS Stock - Upper Circuit Scenario")
simulator.simulate_trade(
    'TCS',
    previous_close=3000,
    prices=[3000, 3200, 3400, 3600, 3400]  # 3600 is upper circuit
)

## Step 6: Running the Production Bot

To run the actual trading bot with live data:

In [ ]:
# PRODUCTION CODE - DO NOT RUN WITHOUT PROPER SETUP
# This is the actual code structure for trading with Kite API

production_code = '''
#!/usr/bin/env python3
"""
Zerodha Kite Circuit Breaker Trading Bot - PRODUCTION
"""

import os
from kiteconnect import KiteConnect
import schedule
import time
import logging

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Get credentials from environment variables (NEVER hardcode!)
API_KEY = os.getenv('KITE_API_KEY')
ACCESS_TOKEN = os.getenv('KITE_ACCESS_TOKEN')

# Initialize Kite
kite = KiteConnect(api_key=API_KEY)
kite.set_access_token(ACCESS_TOKEN)

# Configuration
STOCKS = ['INFY', 'TCS', 'RELIANCE', 'HDFC', 'BAJAJFINSV']
UPPER_CIRCUIT = 20
LOWER_CIRCUIT = 20
QUANTITY = 1

def check_and_trade():
    """Check for circuit breakers and place orders"""
    for symbol in STOCKS:
        try:
            # Get stock quote
            quote = kite.quote(symbols=[symbol])['data'][symbol]
            ltp = quote['last_price']
            prev_close = quote['ohlc']['close']
            
            # Calculate circuit levels
            upper = prev_close * 1.20
            lower = prev_close * 0.80
            
            # Check if circuit hit
            if ltp >= upper * 0.98:
                logger.warning(f"{symbol} hit UPPER CIRCUIT!")
                # Place SELL order
                kite.place_order(
                    variety='regular',
                    exchange='NSE',
                    tradingsymbol=symbol,
                    transaction_type='SELL',
                    quantity=QUANTITY,
                    price=ltp,
                    order_type='limit',
                    product='MIS'
                )
            
            elif ltp <= lower * 1.02:
                logger.warning(f"{symbol} hit LOWER CIRCUIT!")
                # Place BUY order
                kite.place_order(
                    variety='regular',
                    exchange='NSE',
                    tradingsymbol=symbol,
                    transaction_type='BUY',
                    quantity=QUANTITY,
                    price=ltp,
                    order_type='limit',
                    product='MIS'
                )
        
        except Exception as e:
            logger.error(f"Error trading {symbol}: {e}")

# Schedule the check every 2 minutes
schedule.every(2).minutes.do(check_and_trade)

# Keep running
if __name__ == "__main__":
    logger.info("Circuit Trading Bot Started")
    while True:
        schedule.run_pending()
        time.sleep(1)
'''

print("=" * 70)
print("PRODUCTION CODE STRUCTURE")
print("=" * 70)
print(production_code)

print("\n" + "=" * 70)
print("HOW TO RUN:")
print("=" * 70)
print("""
1. Set environment variables:
   Windows (PowerShell):
   $env:KITE_API_KEY = "your_api_key"
   $env:KITE_ACCESS_TOKEN = "your_access_token"
   
   Linux/Mac:
   export KITE_API_KEY="your_api_key"
   export KITE_ACCESS_TOKEN="your_access_token"

2. Run the bot:
   python zerodha_circuit_trader.py

3. Check logs:
   tail -f circuit_trader.log
""")

## Tips & Best Practices for Circuit Trading

### ✅ DO's:
1. **Start Small** - Begin with 1-2 shares per trade
2. **Use Stop Loss** - Protect against unexpected moves (2%)
3. **Monitor Daily** - Watch bot activity in logs
4. **Test First** - Use small quantities before scaling
5. **Keep Logs** - Track all trades for analysis
6. **Use Environment Variables** - Never hardcode API keys
7. **Check Market Status** - Ensure market is open before trading
8. **Use MIS Product** - For intraday trading (4x leverage)

### ❌ DON'Ts:
1. **Don't Trade Without Rules** - Stick to SL and TP
2. **Don't Hardcode Credentials** - Use .env files
3. **Don't Trade Holiday Periods** - Market is closed
4. **Don't Forget Risk Management** - Manage position size
5. **Don't Run Without Internet** - Bot needs live connection
6. **Don't Trade Illiquid Stocks** - Stick to NSE nifty50/100
7. **Don't Ignore Circuit Breakers** - They halt trading for safety
8. **Don't Set High Leverage** - Use conservative quantities

### 📊 Stocks Most Prone to Circuit Breakers:
- Infosys (INFY)
- TCS
- Reliance (RELIANCE)
- HDFC Bank
- IndusInd Bank
- Bajaj Finance (BAJAJFINSV)

## Common Questions & Answers

### Q1: When do circuit breakers trigger?
When a stock moves 20% up or down from previous close during market hours.

### Q2: Can I trade during circuit breaker?
No, trading halts automatically. But our bot buys/sells just before or as it hits.

### Q3: What's the best strategy for circuit trading?
- BUY when lower circuit hits (stock is oversold)
- SELL when upper circuit hits (stock is overbought)
- Use strict SL/TP to manage risk

### Q4: How much capital do I need?
Minimum Rs. 10,000-20,000 per position with Zerodha's 4x MIS margin.

### Q5: Is this profitable?
Yes, but circuit moves are rare (1-2 times per month per stock). Risk is high.

### Q6: What's the success rate?
Depends on risk/reward ratio. With good SL/TP: 50-60% trades profitable.

### Q7: Can I run the bot on my laptop?
Yes, but better to run on a VPS for 24/7 connectivity.

### Q8: What if bot disconnects?
Add reconnection logic and monitor hourly. Better to use VPS.

### Files Created:
- `zerodha_circuit_trader.py` - Main bot (production-ready)
- `README_SETUP.md` - Complete setup guide
- `kite.ipynb` - This educational notebook